<a href="https://colab.research.google.com/github/marcelcerri/Python_biocientistas_unesp/blob/main/Aula_08_Titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Titanic - Previsão sobre os sobreviventes

https://www.kaggle.com/competitions/titanic/data

**Nome das colunas e as informações para entendimento dos dados**

Survival - 0 = No 1 = Yes

Pclass - 1 = 1st, 2 = 2st, 3 = 3st

sex	- Sex

Age	- Age in years

sibsp - # of siblings / spouses aboard the Titanic

parch - # of parents / children aboard the Titanic

ticket	- Ticket number

fare	 - Passenger fare

cabin	 - Cabin number

embarked - Port of Embarkation	 - C = Cherbourg, Q = Queenstown, S = Southampton

In [93]:
import numpy as np
import pandas as pd

In [94]:
test_data = pd.read_csv('test.csv')
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [95]:
train_data = pd.read_csv('train.csv')
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


O que podemos notar é que tem uma coluna a mais no arquivo de treinamento, é a coluna "Survived" em relação ao arquivo de test. A ideia é gerar essa coluna com Machine Learning e postar no Kaggle.

*** 1 for survived, 0 for deceased

In [96]:
women = (train_data.loc[train_data.Sex == 'female']["Survived"])*100
rate_women = sum(women)/len(women)

print(f"% de mulheres que sobreviveram: {rate_women:.2f}")

% de mulheres que sobreviveram: 74.20


In [97]:
men = (train_data.loc[train_data.Sex == 'male']["Survived"])*100
rate_men = sum(men)/len(men)

print(f"% de homens que sobreviveram: {rate_men:.2f}")

% de homens que sobreviveram: 18.89


O Scikit-Learn (frequentemente importado como sklearn) é a principal biblioteca de Machine Learning (Aprendizado de Máquina) em Python. Ele fornece ferramentas simples, eficientes e padronizadas para análise preditiva de dados, servindo como a ponte perfeita entre a manipulação de dados (feita com Pandas) e a criação de modelos de inteligência artificial.

Os modelos do Scikit-Learn são, no fundo, equações matemáticas avançadas. Por causa disso, eles têm duas regras rígidas: eles não entendem texto e eles não sabem lidar com espaços vazios.
 * Tratamentos de Dados Faltantes (Valores Nulos)
 * Transformar Texto em Número: Trocar textos por números simples. Ex: Feminino = 1 e Masculino = 0
 * Separação de X e y (Treino e Teste)

Antes de treinar o modelo, você precisa dividir seus dados de duas formas diferentes:
1. Separar as Perguntas da Resposta (X e y):

1.1 - X (Features): São as variáveis usadas para prever (idade, classe, preço do bilhete).

1.2 - y (Target): É a variável que você quer adivinhar (Sobreviveu: 1 ou 0).
 * Colocar na mesma Escala (Escalonamento / Scaling)

Explicando a função pd.get_dummies - Ele converte variáveis de texto em sistemas binários.

In [98]:
import pandas as pd

sizes = ['Small', 'Medium', 'Large', 'Medium', 'Small', 'Small']
df_dummies = pd.get_dummies(sizes, dtype=int)
print(df_dummies)

   Large  Medium  Small
0      0       0      1
1      0       1      0
2      1       0      0
3      0       1      0
4      0       0      1
5      0       0      1


O Random Forest (Floresta Aleatória) é um dos algoritmos mais populares e poderosos do Scikit-Learn. Ele funciona com base em um conceito chamado "Ensemble" (trabalho em conjunto).

Para entendê-lo, primeiro pense em uma Árvore de Decisão: um fluxograma de perguntas simples de "sim ou não" (Ex: É mulher? Estava na 1ª classe? Tinha irmãos no navio?) que vai afunilando até chegar a uma resposta final (Sobreviveu ou Morreu).
Cada árvore analisa os dados de um ângulo ligeiramente diferente e dá o seu palpite. No final, o modelo faz uma votação: a resposta que a maioria das árvores escolher é a decisão final do algoritmo. Esse "conselho de especialistas" torna o modelo muito mais robusto e à prova de erros.

In [99]:
from sklearn.ensemble import RandomForestClassifier # Importa a biblioteca sklearn - função Random Forest Classifier

y = train_data["Survived"] # É a variável Alvo, é o que queremos prever, a minha saída será a coluna "Survived" do arquivo train

features = ["Pclass", "Sex"] #Quais variáveis escolhemos para treinar o machine learning, estamos treinando com Classe, Sexo

X = pd.get_dummies(train_data[features])

X_test = pd.get_dummies(test_data[features])

model = RandomForestClassifier(n_estimators=5000, max_depth=3, random_state=1)
model.fit(X, y)
predictions = model.predict(X_test)

output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!


Utilizando Sexo e a Classe que a pessoa embarcou obtivemos a pontuação 0.76555

In [100]:
resultado = pd.read_csv('submission.csv')
print(resultado.head())

   PassengerId  Survived
0          892         0
1          893         1
2          894         0
3          895         0
4          896         1


In [101]:
from sklearn.ensemble import RandomForestClassifier # Importa a biblioteca sklearn - função Random Forest Classifier

y = train_data["Survived"] # É a variável Alvo, é o que queremos prever, a minha saída será a coluna "Survived" do arquivo train

features = ["Sex"] #Quais variáveis escolhemos para treinar o machine learning, estamos treinando com Sexo

X = pd.get_dummies(train_data[features])

X_test = pd.get_dummies(test_data[features])

model = RandomForestClassifier(n_estimators=5000, max_depth=3, random_state=1)
model.fit(X, y)
predictions = model.predict(X_test)

output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!


Utilizando apenas Sexo da pessoa que embarcou obtivemos a pontuação 0.76555 - a mesma pontuação anterior.

In [102]:
train_data.info()
print("-"*40)
test_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
----------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Passenger

In [103]:
train_data.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [104]:
train_data[["Sex", "Survived"]].groupby(['Sex'], as_index=False).mean().sort_values(by='Survived', ascending=False)

,Sex,Survived
0,female,0.742038
1,male,0.188908


In [105]:
# Preparação dos Dados (Preprocessing)
# Os modelos de ML só entendem números. Precisamos limpar dados faltantes e transformar textos em números.
# É crucial aplicar as mesmas transformações em AMBOS os datasets (treino e teste).

# Combinar os dataframes para aplicar as transformações de uma só vez
combined_df = pd.concat([train_data.drop('Survived', axis=1), test_data], ignore_index=True)

In [106]:
# Lidando com dados faltantes
# Idade (Age): Preencher com a mediana é uma abordagem simples e robusta.
combined_df['Age'] = combined_df['Age'].fillna(combined_df['Age'].median())

# Tarifa (Fare): O dataset de teste tem uma tarifa faltando. Usaremos a mediana também.
combined_df['Fare'] = combined_df['Fare'].fillna(combined_df['Fare'].median())

# Embarque (Embarked): Preencher com o porto mais comum (a moda).
combined_df['Embarked'] = combined_df['Embarked'].fillna(combined_df['Embarked'].mode()[0])

# Cabine (Cabin): Tem muitos dados faltantes. Por simplicidade, vamos remover essa coluna.
# Usamos errors='ignore' para evitar erros caso a célula seja executada mais de uma vez.
combined_df = combined_df.drop('Cabin', axis=1, errors='ignore')

In [107]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  1309 non-null   int64  
 1   Pclass       1309 non-null   int64  
 2   Name         1309 non-null   object 
 3   Sex          1309 non-null   object 
 4   Age          1309 non-null   float64
 5   SibSp        1309 non-null   int64  
 6   Parch        1309 non-null   int64  
 7   Ticket       1309 non-null   object 
 8   Fare         1309 non-null   float64
 9   Embarked     1309 non-null   object 
dtypes: float64(2), int64(4), object(4)
memory usage: 102.4+ KB


In [108]:
# Convertendo categorias em números
# Gênero (Sex):
combined_df['Sex'] = combined_df['Sex'].map({'male': 0, 'female': 1}).astype(int)

In [109]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  1309 non-null   int64  
 1   Pclass       1309 non-null   int64  
 2   Name         1309 non-null   object 
 3   Sex          1309 non-null   int64  
 4   Age          1309 non-null   float64
 5   SibSp        1309 non-null   int64  
 6   Parch        1309 non-null   int64  
 7   Ticket       1309 non-null   object 
 8   Fare         1309 non-null   float64
 9   Embarked     1309 non-null   object 
dtypes: float64(2), int64(5), object(3)
memory usage: 102.4+ KB


In [110]:
# Porto de Embarque (Embarked): Usaremos "One-Hot Encoding" com pd.get_dummies
combined_df = pd.get_dummies(combined_df, columns=['Embarked'], prefix='Embarked')

In [111]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  1309 non-null   int64  
 1   Pclass       1309 non-null   int64  
 2   Name         1309 non-null   object 
 3   Sex          1309 non-null   int64  
 4   Age          1309 non-null   float64
 5   SibSp        1309 non-null   int64  
 6   Parch        1309 non-null   int64  
 7   Ticket       1309 non-null   object 
 8   Fare         1309 non-null   float64
 9   Embarked_C   1309 non-null   bool   
 10  Embarked_Q   1309 non-null   bool   
 11  Embarked_S   1309 non-null   bool   
dtypes: bool(3), float64(2), int64(5), object(2)
memory usage: 96.0+ KB


In [112]:
# Selecionando as colunas que vamos usar no modelo (features)
# Por simplicidade, vamos remover 'Name', 'Ticket' e 'PassengerId' do treino.
combined_df.drop(['Name', 'Ticket', 'PassengerId'], axis=1, inplace=True)

In [113]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Pclass      1309 non-null   int64  
 1   Sex         1309 non-null   int64  
 2   Age         1309 non-null   float64
 3   SibSp       1309 non-null   int64  
 4   Parch       1309 non-null   int64  
 5   Fare        1309 non-null   float64
 6   Embarked_C  1309 non-null   bool   
 7   Embarked_Q  1309 non-null   bool   
 8   Embarked_S  1309 non-null   bool   
dtypes: bool(3), float64(2), int64(4)
memory usage: 65.3 KB


In [114]:
# Separar os dataframes de volta
train_processed = combined_df.iloc[:len(train_data)]
test_processed = combined_df.iloc[len(train_data):]

In [115]:
#Treinamento do Modelo ---
# Vamos usar um modelo chamado RandomForestClassifier. Ele é poderoso e bom para iniciantes.

# Definir nossos dados de treino
X_train = train_processed
y_train = train_data['Survived'] # Nossa coluna alvo (o que queremos prever)
X_test = test_processed

In [116]:
# Criar e treinar o modelo
model = RandomForestClassifier(n_estimators=1000, max_depth=5, random_state=1)
model.fit(X_train, y_train)

RandomForestClassifier(max_depth=5, n_estimators=1000, random_state=1)

In [117]:
#Fazer Previsões ---
# Agora usamos o modelo treinado para prever a sobrevivência no dataset de teste.
predictions = model.predict(X_test)

In [118]:
# Criar um DataFrame para a submissão
submission = pd.DataFrame({
    "PassengerId": test_data["PassengerId"],
    "Survived": predictions
})

In [119]:
print("\nArquivo 'submission.csv' criado com sucesso!")
print(submission.head())


Arquivo 'submission.csv' criado com sucesso!
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1
